# AHN window diagnostic — GatedDeltaNet, observe-only (Task #1) — **Google Colab**

Environment port. **Same validated experiment/diagnostic**
(branch `saadat-pipeline-validation`, HEAD `902e1bd`), embedded as a base64 tarball and
reconstructed into `/content/ahn-mdc`. No GitHub auth, no repo upload.

Colab's runtime is now **Python 3.13**, but the AHN stack (transformers 4.51, the
Seerkfang FLA fork, the AHN modeling code) is only validated on **≤ 3.12**. This
notebook builds an **isolated Python 3.12 venv at `/content/ahn-py312`** (via `uv`) and
runs the setup + verification + diagnostic **as subprocesses in that venv**. The Colab
3.13 kernel is only used to read the GPU and orchestrate — **nothing is installed into
it except `uv`, and no runtime restart is needed.**

The Juan-approved matched inference config is baked into the bundled
`src/ahnexp/models.py::_force_window` (**sliding_window=256, sliding_window_type=fixed,
ahn_position=prefix, num_attn_sinks=0**, stale `dy_*` deleted). The diagnostic is
**observe-only** — it never writes `model.config`. Exactly **one exact-memory and one
recurrent-memory generation**; same hard gates and JSON fields as the Kaggle run.

## ⚠️ You need an Ampere+ GPU
flash-attn 2.x kernels are **sm_80+ only** and the AHN model calls flash-attention every
decode step. Colab **free = T4 (sm_75) — will not work**. Use **Colab Pro / Pro+** with an
**L4** or **A100** runtime (*Runtime ▸ Change runtime type ▸ GPU*). Cell 1 hard-stops
otherwise.

## Run order (no restart)
| step | cell | runs in | note |
|---|---|---|---|
| A — GPU gate | **1** | 3.13 kernel | hard-stops unless CUDA + compute-capability major ≥ 8; prints exact GPU |
| B — build 3.12 venv | **2** | 3.13 kernel | `pip install uv` (only kernel install) → `/content/ahn-py312`; verifies it reports 3.12 |
| C — reconstruct project | **3** | 3.13 kernel | file writes only |
| D — environment | **4** | 3.12 venv | `SETUP_PY=/content/ahn-py312/bin/python bash setup_kaggle.sh`; torch 2.6/cu126 + prebuilt flash-attn wheel, **no compile**; exits non-zero on any failure |
| E — verify | **5** | 3.12 venv | all checks + a real `flash_attn_func` GPU call |
| F — diagnostic | **6** | 3.12 venv | ~15–20 min first run; **exactly 2 generations** |
| G — JSON | **7** | 3.13 kernel | print the report |

tarball sha256: `174ece2d471910efca7fcd208dcf0e29e6d2e8da3f8864bfe7f9c8f98fef5d26`


## A · Cell 1 — inspect the assigned GPU (hard-stop unless sm_80+)


In [ ]:
import subprocess

name, cap, cuda_ok = None, None, False
try:
    import torch  # Colab kernel torch, used here only to read the device
    cuda_ok = torch.cuda.is_available()
    if cuda_ok:
        name = torch.cuda.get_device_name(0)
        cap = torch.cuda.get_device_capability(0)
except Exception as e:
    print("torch probe failed, using nvidia-smi:", e)

if cap is None:
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,compute_cap", "--format=csv,noheader"]
        ).decode().strip().splitlines()[0]
        n, cc = [x.strip() for x in out.split(",")]
        name, cap = name or n, tuple(int(x) for x in cc.split("."))
        cuda_ok = True
    except Exception as e:
        raise SystemExit(f"STOP: no NVIDIA GPU detected ({e}). "
                         "Runtime > Change runtime type > GPU.")

print(f"GPU model          : {name}")
print(f"compute capability : sm_{cap[0]}{cap[1]}")
print(f"CUDA available     : {cuda_ok}")

if not cuda_ok:
    raise SystemExit("STOP: CUDA is not available. Runtime > Change runtime type > GPU.")
if cap[0] < 8:
    raise SystemExit(
        f"\nSTOP: {name} is sm_{cap[0]}{cap[1]}. flash-attn 2.x kernels are sm_80+ only "
        "(Ampere/Ada/Hopper); the AHN model calls flash-attention every decode step. "
        "Colab free = T4 (sm_75). Switch to Colab Pro/Pro+ and select an L4 or A100 "
        "runtime (Runtime > Change runtime type), then re-run this cell."
    )
print("\nGPU OK (sm_80+). Proceed to Cell 2.")


## B · Cell 2 — build the isolated Python 3.12 venv


In [ ]:
import subprocess, sys, shutil

PY312 = "/content/ahn-py312/bin/python"
VENV  = "/content/ahn-py312"

# `uv` is the ONLY thing installed into the Colab 3.13 kernel (orchestration only).
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv"])
UV = shutil.which("uv") or f"{sys.prefix}/bin/uv"
print("uv:", subprocess.check_output([UV, "--version"]).decode().strip())

subprocess.check_call([UV, "python", "install", "3.12"])
subprocess.check_call([UV, "venv", "--seed", "--python", "3.12", VENV])

# requirement 1: confirm the created interpreter really is 3.12 before continuing
ver = subprocess.check_output([PY312, "-c", "import sys;print('%d.%d.%d' % sys.version_info[:3])"]).decode().strip()
print("isolated interpreter:", PY312, "-> Python", ver)
assert ver.startswith("3.12."), f"expected Python 3.12.x at {PY312}, got {ver}"
print("\n3.12 venv ready. Proceed to Cell 3.")


## C · Cell 3 — reconstruct the minimal project


In [ ]:
import base64, hashlib, io, tarfile, pathlib

ROOT = "/content/ahn-mdc"
EXPECT_SHA = "174ece2d471910efca7fcd208dcf0e29e6d2e8da3f8864bfe7f9c8f98fef5d26"

_BLOB = (
    "..."
    "ff6xPv8/mmYZIQDQAgA="
)

blob = base64.b64decode(_BLOB)
assert hashlib.sha256(blob).hexdigest() == EXPECT_SHA, "tarball checksum mismatch"
pathlib.Path(ROOT).mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(blob), mode="r:gz") as tf:
    tf.extractall(ROOT)

got = sorted(str(p.relative_to(ROOT)) for p in pathlib.Path(ROOT).rglob("*") if p.is_file())
print(f"reconstructed {len(got)} files under {ROOT}")

for need in ("config/experiment.yaml", "config/facts.yaml", "src/ahnexp/models.py",
             "src/ahnexp/dataset.py", "src/ahnexp/evaluate.py",
             "scripts/diag_ahn_window.py", "scripts/setup_kaggle.sh"):
    assert (pathlib.Path(ROOT) / need).is_file(), f"missing {need}"

m = (pathlib.Path(ROOT) / "src/ahnexp/models.py").read_text()
assert 'model.config.sliding_window_type = matched["sliding_window_type"]' in m
assert "model.config.num_attn_sinks = 0" in m
assert '("dy_sliding_window", "dy_num_attn_sinks")' in m
s = (pathlib.Path(ROOT) / "scripts/setup_kaggle.sh").read_text()
assert 'TORCH_VER="2.6.0"' in s and 'cu126' in s and 'SETUP_PY' in s
assert 'FA_VER="2.8.3.post1"' in s
assert 'FLASH_ATTENTION_FORCE_BUILD' not in s
print("OK - matched-config freeze present; setup pins torch 2.6/cu126 + prebuilt flash-attn (no compile).")


## D · Cell 4 — environment (into the 3.12 venv)

Runs the bundled `scripts/setup_kaggle.sh` with `SETUP_PY=/content/ahn-py312/bin/python` so **every install
goes into `/content/ahn-py312`, not the Colab kernel**. It pins `torch==2.6.0` from the
**cu126** (manylinux_2_28 / CXX11-ABI-TRUE) index, removes torchvision/torchaudio,
installs the **prebuilt** `flash_attn-2.8.3.post1` torch2.6 abiTRUE **cp312** wheel by URL
(**no source build**), the Seerkfang `flash-linear-attention` fork, the ByteDance AHN
package (core only), and `numpy`/`pandas`/`pyyaml`/`jinja2`/`pyarrow`/`accelerate`.
`transformers` stays pinned at `4.51.0`. The fail-fast check asserts the interpreter is
3.11/3.12 and that `torch`/`transformers`/`triton`/`fla`/`flash_attn`/`ahn` all import.


In [ ]:
import os, subprocess

env = {**os.environ,
       "SETUP_PY": "/content/ahn-py312/bin/python",
       "AHNEXP_ROOT": "/content/ahn-mdc",
       "AHN_REPO": "/content/AHN"}
rc = subprocess.call(["bash", "/content/ahn-mdc/scripts/setup_kaggle.sh"], env=env)
print("\nsetup exit code:", rc)
assert rc == 0, "setup failed - see the FAIL lines above."
print("environment ready in /content/ahn-py312. Proceed to Cell 5 (no restart).")


## E · Cell 5 — verify the 3.12 environment (real GPU kernel call)


In [ ]:
import subprocess, sys
PY312 = "/content/ahn-py312/bin/python"
ver = subprocess.check_output([PY312, "-c", "import sys;print(sys.version_info[0],sys.version_info[1])"]).decode().split()
assert ver == ["3", "12"], f"{PY312} is Python {ver}, expected 3.12 - re-run Cell 2"
VERIFY = 'import sys\nassert sys.version_info[:2] == (3, 12), f"verify ran under Python {sys.version_info[:2]}, expected 3.12"\nimport torch, transformers, importlib\nprint("interpreter        ", sys.executable, "Python", sys.version.split()[0])\nprint("torch              ", torch.__version__, "| cuda", torch.version.cuda)\nprint("cuda available     ", torch.cuda.is_available())\nassert torch.cuda.is_available(), "no CUDA GPU in the 3.12 venv"\nname = torch.cuda.get_device_name(0); cap = torch.cuda.get_device_capability(0)\nprint("gpu                ", name, f"sm_{cap[0]}{cap[1]}")\nprint("cxx11 abi          ", torch.compiled_with_cxx11_abi())\nprint("transformers       ", transformers.__version__)\nassert torch.__version__.startswith("2.6."), torch.__version__\nassert torch.compiled_with_cxx11_abi(), "torch is not CXX11-ABI-TRUE"\nassert transformers.__version__ == "4.51.0", transformers.__version__\nassert cap[0] >= 8, f"GPU sm_{cap[0]}{cap[1]} - flash-attn 2.x needs sm_80+"\nfor _m in ("triton", "fla", "flash_attn", "ahn.transformer.qwen2_ahn"):\n    _x = importlib.import_module(_m)\n    print("import %-30s OK  %s" % (_m, getattr(_x, "__version__", "")))\nfrom flash_attn import flash_attn_func\nq = torch.randn(1, 8, 2, 16, dtype=torch.float16, device="cuda")\no = flash_attn_func(q, q, q, causal=True)\nassert tuple(o.shape) == (1, 8, 2, 16)\nprint("flash_attn_func on GPU: OK", tuple(o.shape))\nfrom ahn.transformer.qwen2_ahn import register_customized_qwen2\nregister_customized_qwen2()\nprint("register_customized_qwen2: OK")\nprint("\\nENV VERIFIED - safe to run Cell 6.")\n'
rc = subprocess.call([PY312, "-c", VERIFY])
print("\nverify exit code:", rc)
assert rc == 0, "environment verification failed - see above."


## F · Cell 6 — run the observe-only diagnostic (with the 3.12 interpreter)

Loads **only GatedDeltaNet**, merges weights once (~6 GB base download, cached under
`/content/ahn-mdc/merged_ckpt`), verifies the custom AHN class + `.ahn` params, builds two
trajectories straddling W=256, **hard-stops if the recurrent one does not cross W**, then
runs **exactly one exact-memory and one recurrent-memory generation**. Observe-only — it
does not modify `model.config`. Same hard gates and output fields.


In [ ]:
import subprocess
PY312 = "/content/ahn-py312/bin/python"
ver = subprocess.check_output([PY312, "-c", "import sys;print(sys.version_info[0],sys.version_info[1])"]).decode().split()
assert ver == ["3", "12"], f"{PY312} is Python {ver}, expected 3.12"
cmd = [PY312, "/content/ahn-mdc/scripts/diag_ahn_window.py", "--repo", "/content/ahn-mdc", "--ahn-repo", "/content/AHN"]
print(" ".join(cmd), "\n")
rc = subprocess.call(cmd)
print("\ndiagnostic exit code:", rc, "(0 = PASSED, 2 = INCONCLUSIVE, 1 = hard-stop/error)")


## G · Cell 7 — print the diagnostic JSON


In [ ]:
import pathlib
p = pathlib.Path("/content/ahn-mdc/outputs/diag_ahn_window.json")
print(p.read_text() if p.is_file() else
      "no JSON - the run hard-stopped early; paste the Cell 6 output above.")


## What to paste back for review

Full output of **Cell 1** (GPU), **Cell 4** (environment tail), **Cell 5** (verify),
**Cell 6** (diagnostic), and the **Cell 7** JSON. Key checks:

* Cell 1 / Cell 5: `GPU` is L4 (`sm_89`) or A100 (`sm_80`)
* Cell 4 tail: `interpreter /content/ahn-py312/bin/python  Python 3.12.x`, then
  `SETUP OK (torch 2.6.0 cxx11abi=TRUE, flash-attn 2.8.3.post1 prebuilt, cp312)`
* Cell 5: `torch 2.6.x`, `cxx11 abi True`, `transformers 4.51.0`, `flash_attn_func on GPU: OK`, `register_customized_qwen2: OK`
* `[2]` checkpoint pre-override: `sliding_window 256`, `sliding_window_type random`, `ahn_position random`
* `[3]` after `models.load`: `sliding_window 256`, **`sliding_window_type fixed`**, **`ahn_position prefix`**, **`num_attn_sinks 0`**, `dy_* <unset>`, `model.training False`, `effective_window 256`
* `[3b]` `model class ahn.transformer.qwen2_ahn.qwen2_ahn.Qwen2ForCausalLM`; `generation_config.use_cache True`
* `[2] AHN modules` `36 x Qwen2MemDecoderLayer`, `layer[0] .ahn = BaseAHN / fn=GatedDeltaNet`
* `[6]` the two `model_tokens_after_target` (≈49 and ≈2100)
* `HARD GATE` line = `PASS`
* `[9]` both trial rows: `prediction`, `answer_canonical`, `correct`, `malformed`, `abstained`, `n_new_tokens`, **`ahn_layer0_num_cached_tokens`** (0 exact / ~1850+ recurrent), **`ahn_kernel_forward_calls`** (0 exact / ≥1 recurrent)
* `[10]` verdict block + `DIAGNOSTIC PASSED` / `INCONCLUSIVE`

If Cell 6 exits early, paste what printed — the hard gate stopping is a valid result. **Do not run the experimental grid regardless of outcome.**
